<a href="https://colab.research.google.com/github/MariaMuu/Thesis/blob/main/nl_to_sparql_thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Required Packages

In [1]:
# Install required libraries in Colab
!pip install transformers
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 51.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Install the OpenAI library
!pip install openai

# Set Up OpenAI API Key & Define Your Fine-Tuned Model Call

In [37]:
import openai
from google.colab import userdata

# Set your OpenAI API key securely
openai.api_key = userdata.get('OPENAI_API_KEY') # No longer needed when passing api_key to client constructor

# Replace with the actual name of your fine-tuned model
fine_tuned_model_name = "gpt-4.1"

# Example function to get a response from your fine-tuned model
def get_response_from_openai_model(prompt):
    try:
        # Get the API key securely
        api_key = userdata.get('OPENAI_API_KEY')
        client = openai.OpenAI(api_key=api_key) # Initialize client here as well if needed

        response = client.completions.create( # Use client.completions.create for completion models
            model=fine_tuned_model_name,
            prompt=prompt,
            max_tokens=150
        )
        return response.choices[0].text.strip()
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

# Example usage:
# question = "What is the capital of France?"
# sparql_query = get_response_from_openai_model(question)
# print(sparql_query)

# Wikidata Entity & Property Mapping Functions

In [38]:
import openai
from google.colab import userdata

def get_sparql_from_openai(question, model_name):
    # Get the API key securely
    api_key = userdata.get('OPENAI_API_KEY')
    client = openai.OpenAI(api_key=api_key) # Initialize the client with the API key

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You translate English questions into SPARQL."},
            {"role": "user", "content": question}
        ],
        temperature=0.1,
        max_tokens=500
    )
    sparql_query = response.choices[0].message.content.strip()
    return sparql_query

# Full Pipeline Function

In [39]:
import requests
import spacy

nlp = spacy.load("en_core_web_sm")
WIKIDATA_API_URL = "https://www.wikidata.org/w/api.php"

def search_entity(term):
    params = {
        'action': 'wbsearchentities',
        'format': 'json',
        'language': 'en',
        'search': term
    }
    response = requests.get(WIKIDATA_API_URL, params=params).json()
    if response.get('search'):
        return response['search'][0]['id']
    return None

def search_property(term):
    params = {
        'action': 'wbsearchentities',
        'format': 'json',
        'language': 'en',
        'type': 'property',
        'search': term
    }
    response = requests.get(WIKIDATA_API_URL, params=params).json()
    if response.get('search'):
        return response['search'][0]['id']
    return None

def extract_entities_verbs(text):
    doc = nlp(text)
    entities = list(set([ent.text for ent in doc.ents]))
    verbs = list(set([token.lemma_ for token in doc if token.pos_ == 'VERB']))
    return entities, verbs

In [40]:
def ask_question_with_mapping(question, model_name):
    # 1. Get SPARQL from OpenAI fine-tuned model
    sparql = get_sparql_from_openai(question, model_name)
    print("\n🔹 Model-generated SPARQL:\n", sparql)

    # 2. Extract entities and verbs from question
    entities, verbs = extract_entities_verbs(question)

    # 3. Map entities to Q-IDs
    print("\n🔹 Detected Entities:")
    entity_map = {}
    for entity in entities:
        qid = search_entity(entity)
        if qid:
            entity_map[entity] = qid
            print(f"✅ {entity} → {qid}")

    # 4. Map verbs to P-IDs
    print("\n🔹 Detected Relations:")
    relation_map = {}
    for verb in verbs:
        pid = search_property(verb)
        if pid:
            relation_map[verb] = pid
            print(f"✅ {verb} → {pid}")

    # 5. (Optional) Here you could auto-correct the SPARQL with these IDs

    return sparql, entity_map, relation_map

In [45]:
# model_name = "ft:gpt-4.1-mini-2025-04-14:personal-project::BleQwnO7" # This was for the old Completion API

# Assuming your fine-tuned model is a chat completion model,
# use the appropriate model name here.
# If you fine-tuned a Completion model, you might need to adapt the function call in get_sparql_from_openai
model_name = "gpt-4.1" # Example chat fine-tuned model name

question = "tell me some cryptocurrency coins?"
sparql, entity_map, relation_map = ask_question_with_mapping(question, model_name)


🔹 Model-generated SPARQL:
 SELECT DISTINCT ?obj WHERE { wd:Q1343071 wdt:P2416 ?obj . ?obj wdt:P31 wd:Q1343071}

🔹 Detected Entities:

🔹 Detected Relations:
